# Backup, Recovery, and Replication

Disks fail, people run `DELETE` without `WHERE`, and ransomware encrypts servers. A backup is only useful if you can **restore** it, within the time the business can wait and with the amount of data loss it can accept.

**After this lesson you can:**

1. Define RPO and RTO and choose a backup strategy.
2. Back up and restore a database with `pg_dump` and `pg_restore`.
3. Recover from an accidental deletion.
4. Explain WAL, point-in-time recovery and replication.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine, text

sys.path.insert(0, str(Path.cwd().parent / "tools"))
import datasets                                   # course helper: downloads and loads the sample data

engine = create_engine(datasets.ensure_chinook())  # starts PostgreSQL in Docker and loads Chinook on first use

def q(sql: str, **params) -> pd.DataFrame:
    """Run a SQL query and return the result as a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

import subprocess
import tempfile
import time

COMPOSE = ["docker", "compose", "-f", str(datasets.COMPOSE)]
WORK = Path(tempfile.mkdtemp())

def in_container(*cmd: str, stdin: bytes | None = None) -> bytes:
    """Run a PostgreSQL client tool inside the database container and return its stdout."""
    r = subprocess.run([*COMPOSE, "exec", "-T", "postgres", *cmd], input=stdin, capture_output=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.decode()[-500:])
    return r.stdout

## 1. RPO and RTO

| Term | Question | Example target |
| :--- | :--- | :--- |
| **RPO** (recovery point objective) | how much data may we lose? | at most 5 minutes |
| **RTO** (recovery time objective) | how long may the restore take? | at most 1 hour |

| Strategy | Typical RPO | Notes |
| :--- | :--- | :--- |
| nightly logical dump (`pg_dump`) | up to 24 h | simple, portable between versions |
| physical base backup + WAL archive | seconds to minutes | point-in-time recovery (PITR) |
| streaming replica | close to 0 | protects against server failure, **not** against a bad `DELETE` (the replica deletes too) |

The **3-2-1 rule**: 3 copies of the data, on 2 different media, 1 copy off-site (and ideally one copy that nobody can modify or delete).

## 2. Logical Backup with `pg_dump`

`pg_dump` writes a consistent snapshot of one database while it is in use. Formats:

- **plain** (`-Fp`): a SQL script, readable and restorable with `psql`,
- **custom** (`-Fc`): compressed, restorable with `pg_restore`, which can restore single tables and run in parallel.

In [2]:
sizes = {}
for fmt in ("p", "c"):
    t = time.perf_counter()
    data = in_container("pg_dump", "-U", "course", f"-F{fmt}", "chinook")
    (WORK / f"chinook.{fmt}.dump").write_bytes(data)
    sizes["plain SQL" if fmt == "p" else "custom (compressed)"] = (len(data) / 1e6, time.perf_counter() - t)
for name, (mb, s) in sizes.items():
    print(f"{name:<20} {mb:6.2f} MB in {s:.1f} s")
plain = (WORK / "chinook.p.dump").read_text(encoding="utf-8")
print("\nfirst CREATE TABLE statement in the plain dump:")
start = plain.index("CREATE TABLE")
print(plain[start:plain.index(");", start) + 2])

plain SQL              0.42 MB in 1.8 s
custom (compressed)    0.16 MB in 0.5 s

first CREATE TABLE statement in the plain dump:
CREATE TABLE public.album (
    album_id integer NOT NULL,
    title character varying(160) NOT NULL,
    artist_id integer NOT NULL
);


## 3. Restore into a New Database

**Test your restores.** A backup that was never restored is only a hope. The cell creates an empty database `chinook_restore`, restores the custom dump into it, and compares row counts with the original.

In [3]:
in_container("dropdb", "-U", "course", "--if-exists", "chinook_restore")
in_container("createdb", "-U", "course", "chinook_restore")
t = time.perf_counter()
in_container("pg_restore", "-U", "course", "-d", "chinook_restore", "--no-owner", stdin=(WORK / "chinook.c.dump").read_bytes())
restore_s = time.perf_counter() - t
restored = create_engine(datasets.POSTGRES_URL.format(db="chinook_restore"))

tables = ["artist", "album", "track", "customer", "invoice", "invoice_line"]
check = pd.DataFrame({
    "original": [q(f"SELECT count(*) AS n FROM {t}")["n"][0] for t in tables],
    "restored": [pd.read_sql(text(f"SELECT count(*) AS n FROM {t}"), restored)["n"][0] for t in tables],
}, index=tables)
check["match"] = check["original"] == check["restored"]
print(f"restore took {restore_s:.1f} s")
check

restore took 1.7 s


,original,restored,match
artist,275,275,True
album,347,347,True
track,3503,3503,True
customer,59,59,True
invoice,412,412,True
invoice_line,2240,2240,True


## 4. Recover from an Accidental Delete

A classic incident: somebody deletes rows by mistake. With a logical dump, you restore the affected table from the backup into a separate database and copy back only the missing rows, so the changes made after the backup stay.

In [4]:
with restored.begin() as conn:                                   # the incident (in the restored copy, not in Chinook)
    deleted = conn.execute(text("DELETE FROM invoice_line WHERE invoice_id IN "
                                "(SELECT invoice_id FROM invoice WHERE billing_country = 'Germany')")).rowcount
    conn.execute(text("INSERT INTO genre (genre_id, name) VALUES (26, 'New Genre After Backup')"))   # a later, valid change
print(f"incident: {deleted} invoice lines deleted")

# recovery: restore only the invoice_line table from the backup into a staging database
in_container("dropdb", "-U", "course", "--if-exists", "chinook_staging")
in_container("createdb", "-U", "course", "chinook_staging")
in_container("pg_restore", "-U", "course", "-d", "chinook_staging", "--no-owner", "-t", "invoice_line",
             stdin=(WORK / "chinook.c.dump").read_bytes())
staging = create_engine(datasets.POSTGRES_URL.format(db="chinook_staging"))
backup_lines = pd.read_sql(text("SELECT * FROM invoice_line"), staging)
current_ids = set(pd.read_sql(text("SELECT invoice_line_id FROM invoice_line"), restored)["invoice_line_id"])
missing = backup_lines[~backup_lines["invoice_line_id"].isin(current_ids)]
missing.to_sql("invoice_line", restored, if_exists="append", index=False)

after = pd.read_sql(text("SELECT (SELECT count(*) FROM invoice_line) AS lines, "
                         "(SELECT count(*) FROM genre WHERE name LIKE 'New Genre%') AS later_change_kept"), restored)
print(f"rows copied back: {len(missing)}")
after

incident: 152 invoice lines deleted


rows copied back: 152


,lines,later_change_kept
0,2240,1


## 5. WAL, Point-in-Time Recovery and Replication

PostgreSQL writes every change first to the **write-ahead log (WAL)**, then to the data files. This gives durability (Lesson 11) and enables two more tools:

- **Point-in-time recovery (PITR):** take a physical base backup (`pg_basebackup`) and archive every WAL file. To recover, restore the base backup and replay WAL up to a chosen moment, for example one second before the bad `DELETE`.
- **Streaming replication:** a standby server receives WAL continuously and replays it. It can serve read-only queries and take over if the primary fails (**failover**).

The cell shows the WAL settings of the course server and how WAL grows during writes.

In [5]:
settings = q("""SELECT name, setting FROM pg_settings
                WHERE name IN ('wal_level', 'archive_mode', 'max_wal_senders', 'synchronous_commit', 'max_wal_size')""")
print(settings.to_string(index=False))

def wal_lsn():
    return q("SELECT pg_current_wal_lsn()::text AS lsn")["lsn"][0]

lab = create_engine(datasets.POSTGRES_URL.format(db="course"))
start = wal_lsn()
with lab.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS wal_demo; CREATE TABLE wal_demo AS SELECT g, md5(g::text) FROM generate_series(1, 200000) g"))
end = wal_lsn()
wal_bytes = q("SELECT pg_wal_lsn_diff(:e, :s) AS b", e=end, s=start)["b"][0]
print(f"\nWAL generated by writing 200,000 rows: {wal_bytes / 1e6:.1f} MB  (LSN {start} -> {end})")
with lab.begin() as conn:
    conn.execute(text("DROP TABLE wal_demo"))

              name setting
      archive_mode     off
   max_wal_senders      10
      max_wal_size    1024
synchronous_commit      on
         wal_level replica



WAL generated by writing 200,000 rows: 19.3 MB  (LSN 0/4A139078 -> 0/4B39C428)


In [6]:
for eng in (restored, staging):
    eng.dispose()
in_container("dropdb", "-U", "course", "--if-exists", "chinook_restore")
in_container("dropdb", "-U", "course", "--if-exists", "chinook_staging")
print("temporary databases removed")

temporary databases removed


Managed cloud databases (Amazon RDS, Cloud SQL, Azure) automate base backups, WAL archiving and replicas; you choose the retention period and test the restore. Tools such as **pgBackRest** and **Barman** do the same for self-managed servers.

*Insight: Define RPO and RTO with the business first, then choose the backup method. Automate backups, keep one copy off-site and immutable, and restore a backup on a schedule: an untested backup is not a backup.*

## Real-World Use Case or Analogy

Backups are like **spare house keys**. One spare in the house does not help if the house burns down (keep one off-site). A replica is like a twin who copies everything you do: helpful if you are ill, useless if you throw the keys into the sea, because the twin does the same. Point-in-time recovery is a **video recording** of your day that you can rewind to the second before the mistake.

Real incidents show the value of tested restores: several well-known companies have lost hours of production data because their backup jobs had silently failed for months.